In [1]:
import json, os
from pathlib import Path
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig


PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm\parser")
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = PROJECT / "checkpoints" / "qwen1.5b-qlora-v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

torch: 2.11.0+cu128 cuda: True


In [2]:
SYSTEM_PROMPT = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.

Output ONLY the JSON. No explanation, no prose."""

def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]

train_raw = load_jsonl(PROJECT / "data" / "train.jsonl")
test_raw  = load_jsonl(PROJECT / "data" / "test.jsonl")

def to_chat(ex):
    user = f'Options: {ex["options"]}\nUser: {ex["utterance"]}'
    assistant = json.dumps(ex["label"])
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user},
            {"role": "assistant", "content": assistant},
        ]
    }

train_ds = Dataset.from_list([to_chat(e) for e in train_raw])
test_ds  = Dataset.from_list([to_chat(e) for e in test_raw])
print("train:", len(train_ds), "test:", len(test_ds))
print("sample:", train_ds[0])

train: 1278 test: 210
sample: {'messages': [{'role': 'system', 'content': 'You are a parser that converts a user\'s natural language response into a subset of the shown options.\n\nThe user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.\n\nOutput format (JSON only, nothing else):\n- A JSON list of the favored labels, e.g. ["A", "B"]\n- [] if the user explicitly rejects ALL options ("none of these", "all wrong")\n- "*" if the utterance is off-topic OR expresses no usable preference ("I don\'t know", "they all look the same", "I love football")\n\nRules:\n- Any positive signal about an option means it goes in the list.\n- "X is better than Y" endorses only X, not Y.\n- "X and Y are both good, X is better" endorses both X and Y.\n- Negations like "not D" or "anything but B" mean the remaining options go in the list.\n- Questions like "is it A?" are treated as tentative endorsem

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda",
)
model = prepare_model_for_kbit_training(model)
print("model loaded. VRAM (GB):", torch.cuda.memory_allocated() / 1e9)

W0811 18:36:31.943000 19692 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


model loaded. VRAM (GB): 1.620461568


In [4]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [5]:
sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=512,
    packing=False,
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/1278 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1278 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/210 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/210 [00:00<?, ? examples/s]

In [6]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.063179,0.061382,0.066883,358706.000000,0.987044
2,0.053793,0.055566,0.054978,717412.000000,0.988039
3,0.049973,0.054840,0.051553,1076118.000000,0.988021


TrainOutput(global_step=240, training_loss=0.16803004927933216, metrics={'train_runtime': 1149.9935, 'train_samples_per_second': 3.334, 'train_steps_per_second': 0.209, 'total_flos': 8670917502670848.0, 'train_loss': 0.16803004927933216, 'epoch': 3.0})

In [7]:
for n, p in model.named_parameters():
    if p.requires_grad:
        print(n, p.dtype)
        break

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.bfloat16


In [8]:
from peft import PeftModel

# Base model in 4-bit
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda",
)

# Load adapter
adapter_path = str(OUTPUT_DIR / "checkpoint-240")
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_model.eval()
print("adapter loaded from", adapter_path)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


adapter loaded from C:\Users\shlok\projects\ddp-llm\parser\checkpoints\qwen1.5b-qlora-v1\checkpoint-240


In [9]:
import gc, torch
try:
    del trainer, model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("VRAM after cleanup:", torch.cuda.memory_allocated() / 1e9, "GB")

VRAM after cleanup: 2.179522048 GB


In [10]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda",
)
adapter_path = str(OUTPUT_DIR / "checkpoint-240")
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_model.eval()
print("adapter loaded from", adapter_path)
print("VRAM (GB):", torch.cuda.memory_allocated() / 1e9)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

adapter loaded from C:\Users\shlok\projects\ddp-llm\parser\checkpoints\qwen1.5b-qlora-v1\checkpoint-240
VRAM (GB): 2.866070528


In [11]:
def generate_ft(messages, max_new_tokens=40):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")
    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

test_cases = [
    ("A and C look good", '["A", "C"]'),
    ("not D", '["A", "B", "C"]'),
    ("I don't know", '"*"'),
    ("none of these work", '[]'),
    ("A is better than B", '["A"]'),
    ("what's for lunch", '"*"'),
    ("hate all of them", '[]'),
    ("A great, B bad, C great, D bad", '["A", "C"]'),
]

for utt, expected in test_cases:
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: ["A", "B", "C", "D"]\nUser: {utt}'},
    ]
    got = generate_ft(msgs)
    match = "✓" if got.strip() == expected else "✗"
    print(f"{match} {utt!r}\n   expected: {expected}\n   got:      {got}\n")

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✓ 'A and C look good'
   expected: ["A", "C"]
   got:      ["A", "C"]

✓ 'not D'
   expected: ["A", "B", "C"]
   got:      ["A", "B", "C"]

✓ "I don't know"
   expected: "*"
   got:      "*"

✓ 'none of these work'
   expected: []
   got:      []

✓ 'A is better than B'
   expected: ["A"]
   got:      ["A"]

✓ "what's for lunch"
   expected: "*"
   got:      "*"

✓ 'hate all of them'
   expected: []
   got:      []

✓ 'A great, B bad, C great, D bad'
   expected: ["A", "C"]
   got:      ["A", "C"]



In [12]:
from tqdm import tqdm

def parse_output(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return None
    if parsed == "*" or (isinstance(parsed, list) and all(x in ["A","B","C","D"] for x in parsed)):
        return parsed
    return None

def labels_equal(a, b):
    if a == "*" or b == "*":
        return a == b
    if isinstance(a, list) and isinstance(b, list):
        return set(a) == set(b)
    return False

test_examples = load_jsonl(PROJECT / "data" / "test.jsonl")

results = []
for ex in tqdm(test_examples):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: {ex["options"]}\nUser: {ex["utterance"]}'},
    ]
    raw = generate_ft(msgs)
    parsed = parse_output(raw)
    correct = parsed is not None and labels_equal(parsed, ex["label"])
    results.append({
        "utterance": ex["utterance"],
        "category": ex.get("category", "unknown"),
        "gold": ex["label"],
        "raw": raw,
        "parsed": parsed,
        "correct": correct,
        "valid_format": parsed is not None,
    })

n = len(results)
n_valid = sum(r["valid_format"] for r in results)
n_correct = sum(r["correct"] for r in results)
print(f"\nformat validity: {n_valid}/{n} = {n_valid/n:.1%}")
print(f"exact match:     {n_correct}/{n} = {n_correct/n:.1%}")

from collections import defaultdict
by_cat = defaultdict(lambda: [0, 0])
for r in results:
    by_cat[r["category"]][0] += 1
    by_cat[r["category"]][1] += int(r["correct"])
print("\nper category:")
for cat, (total, correct) in sorted(by_cat.items()):
    print(f"  {cat}: {correct}/{total} = {correct/total:.1%}")

100%|██████████| 210/210 [02:05<00:00,  1.68it/s]


format validity: 210/210 = 100.0%
exact match:     208/210 = 99.0%

per category:
  comparative: 26/26 = 100.0%
  mixed_sentiment: 27/28 = 96.4%
  multi_positive: 12/13 = 92.3%
  negation: 61/61 = 100.0%
  off_topic: 22/22 = 100.0%
  reject_all: 29/29 = 100.0%
  single_positive: 9/9 = 100.0%
  uncertainty: 22/22 = 100.0%


In [13]:
print("=== FAILURES ===")
for r in results:
    if not r["correct"]:
        print(f"[{r['category']}] {r['utterance']!r}")
        print(f"   gold:   {r['gold']}")
        print(f"   parsed: {r['parsed']}")
        print(f"   raw:    {r['raw']!r}")
        print()

=== FAILURES ===
[mixed_sentiment] 'A perfect, B terrible, C okay, D terrible'
   gold:   ['A', 'C']
   parsed: ['A']
   raw:    '["A"]'

[multi_positive] 'A is okay but B is closer'
   gold:   ['A', 'B']
   parsed: ['B']
   raw:    '["B"]'



In [14]:
import gc, torch
try:
    del base, ft_model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("VRAM after cleanup:", torch.cuda.memory_allocated() / 1e9, "GB")

VRAM after cleanup: 0.95353344 GB


In [15]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda",
)
adapter_path = str(OUTPUT_DIR / "checkpoint-240")
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_model.eval()
print("adapter loaded from", adapter_path)
print("VRAM (GB):", torch.cuda.memory_allocated() / 1e9)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

adapter loaded from C:\Users\shlok\projects\ddp-llm\parser\checkpoints\qwen1.5b-qlora-v1\checkpoint-240
VRAM (GB): 2.181209088


In [16]:
import json
from pathlib import Path
from collections import Counter

PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm\parser")
train = [json.loads(l) for l in (PROJECT / "data" / "train.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
test  = [json.loads(l) for l in (PROJECT / "data" / "test.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]

print("train negation:", sum(1 for e in train if e.get("category") == "negation"))
print("test  negation:", sum(1 for e in test  if e.get("category") == "negation"))

train negation: 368
test  negation: 61


In [17]:
neg_train = [e for e in train if e.get("category") == "negation"]

def has_except_all_wrong(u):
    u = u.lower()
    return "except" in u and any(w in u for w in ["all wrong", "all off", "everything is", "everything's", "the rest are wrong", "all are off", "everything else is wrong", "everything else is off"])

def has_ordinal(u):
    u = u.lower()
    ordinals = ["first one", "second one", "third one", "fourth one", "the first", "the second", "the third", "the fourth"]
    return any(o in u for o in ordinals)

print(f"total negation train: {len(neg_train)}")
print(f"  'except X, all wrong'-shape: {sum(1 for e in neg_train if has_except_all_wrong(e['utterance']))}")
print(f"  ordinal references: {sum(1 for e in neg_train if has_ordinal(e['utterance']))}")

print("\nsample 'except X'-ish examples:")
for e in neg_train:
    if "except" in e["utterance"].lower():
        print(f"  {e['utterance']!r} -> {e['label']}")

total negation train: 368
  'except X, all wrong'-shape: 20
  ordinal references: 64

sample 'except X'-ish examples:
  'everything except the third is wrong' -> ['C']
  'everything except B is a mistake' -> ['B']
  'everything except B and D is a candidate' -> ['A', 'C']
  'everything except the first is wrong' -> ['A']
  'all of them except none, A B C and D all decent' -> ['A', 'B', 'C', 'D']
  'everyone except A is a miss' -> ['A']
  'all wrong except D' -> ['D']
  'all wrong except C' -> ['C']
  'except A, all are off' -> ['A']
  'except for C, everything is a miss' -> ['C']
  "except A they're all wrong" -> ['A']
  "everything's wrong except B" -> ['B']
  'all off except A' -> ['A']
  'anything except D' -> ['A', 'B', 'C']
  'all wrong except B' -> ['B']
  'everything except D is a candidate' -> ['A', 'B', 'C']
  'except C, all are off' -> ['C']
  'the only exception is A, rest are misses' -> ['A']
  'except for B' -> ['A', 'C', 'D']
  'the only exception is B, rest are misses' -

In [18]:
import json
from pathlib import Path
from collections import Counter

PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm\parser")
synth = [json.loads(l) for l in (PROJECT / "data" / "synthetic_raw.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
print(f"total in synthetic_raw: {len(synth)}")
print("by category:", Counter(e.get("category", "unknown") for e in synth))

total in synthetic_raw: 1446
by category: Counter({'negation': 426, 'mixed_sentiment': 200, 'reject_all': 200, 'comparative': 180, 'uncertainty': 150, 'off_topic': 150, 'multi_positive': 80, 'single_positive': 60})


In [19]:
import json
from pathlib import Path

PATH = Path(r"C:\Users\shlok\projects\ddp-llm\parser\data\synthetic_raw.jsonl")

kept, dropped = 0, 0
lines_out = []
with open(PATH, "r", encoding="utf-8") as f:
    for line in f:
        ex = json.loads(line)
        if ex.get("category", "unknown") == "unknown":
            dropped += 1
            continue
        lines_out.append(line)
        kept += 1

with open(PATH, "w", encoding="utf-8") as f:
    f.writelines(lines_out)

print(f"kept {kept}, dropped {dropped}")

kept 1446, dropped 0


In [20]:
import os; print(os.getcwd())

C:\Users\shlok\projects\ddp-llm\parser\notebooks


In [21]:
import json
from pathlib import Path

PATH = Path(r"C:\Users\shlok\projects\ddp-llm\parser\data\train.jsonl")

except_examples = []
with open(PATH, "r", encoding="utf-8") as f:
    for line in f:
        ex = json.loads(line)
        u = ex["utterance"].lower()
        if any(k in u for k in ["except", "aside from", "everyone except", "everything except"]):
            except_examples.append(ex)

print(f"total 'except'-shape examples in train: {len(except_examples)}")
print(f"with single-element label: {sum(1 for e in except_examples if isinstance(e['label'], list) and len(e['label']) == 1)}")
print(f"with multi-element label: {sum(1 for e in except_examples if isinstance(e['label'], list) and len(e['label']) > 1)}")
print("\nsamples:")
for e in except_examples[:8]:
    print(f"  {e['utterance']!r} -> {e['label']}")

total 'except'-shape examples in train: 48
with single-element label: 42
with multi-element label: 6

samples:
  'everything except the third is wrong' -> ['C']
  'everything except B is a mistake' -> ['B']
  'everything except B and D is a candidate' -> ['A', 'C']
  'aside from D, none work' -> ['D']
  'everything except the first is wrong' -> ['A']
  'all of them except none, A B C and D all decent' -> ['A', 'B', 'C', 'D']
  'aside from A, none work' -> ['A']
  'everyone except A is a miss' -> ['A']


In [22]:
print(f"MODEL_NAME: {MODEL_NAME}")
print(f"adapter path: {adapter_path}")

MODEL_NAME: Qwen/Qwen2.5-1.5B-Instruct
adapter path: C:\Users\shlok\projects\ddp-llm\parser\checkpoints\qwen1.5b-qlora-v1\checkpoint-240


In [23]:
# OOD generalization test — 1.5B v1.5
import json, re

def parse_output(text, valid_letters=None):
    if valid_letters is None:
        valid_letters = ["A", "B", "C", "D"]
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return None
    if parsed == "*":
        return "*"
    if isinstance(parsed, list) and all(x in valid_letters for x in parsed):
        return parsed
    return None

def generate_ft(messages, max_new_tokens=40):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")
    with torch.no_grad():
        out = ft_model.generate(**inputs, max_new_tokens=max_new_tokens,
                                do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# OOD test cases from session 3
tests = [
    # Novel utterances
    (["A","B","C","D"], "eh, only B feels right", ["B"]),
    (["A","B","C","D"], "throw out everything but the third one", ["C"]),
    (["A","B","C","D"], "hard pass on all four", []),
    (["A","B","C","D"], "A? maybe. dunno.", None),  # ambiguous
    (["A","B","C","D"], "the last two are garbage", ["A","B"]),
    (["A","B","C","D"], "B leaves D in the dust", ["B"]),
    (["A","B","C","D"], "meh whatever pick anything", "*"),
    (["A","B","C","D"], "the second and fourth ones look promising", ["B","D"]),
    (["A","B","C","D"], "A is decent, B slightly better, C and D no", ["A","B"]),
    (["A","B","C","D"], "give me option C or nothing", ["C"]),

    # Variable N with unseen letters
    (["A","B","C","D","E"], "E is good", ["E"]),
    (["A","B","C","D","E"], "A and E are best", ["A","E"]),
    (["A","B","C","D","E"], "everything except E is wrong", ["E"]),
    (["A","B","C","D","E"], "the fifth one is good", ["E"]),
    (["A","B","C","D","E"], "not E, not D", ["A","B","C"]),
    (["A","B","C","D","E","F"], "F is the best", ["F"]),
    (["A","B","C","D","E","F","G"], "the last one only", ["G"]),
    (["A","B","C","D","E","F"], "none of these", []),
    (["A","B","C","D","E","F"], "I don't know", "*"),
    (["A","B","C","D","E","F","G"], "only G matters", ["G"]),
]

correct = 0
ambiguous = 0
print(f"{'options':<25s} {'utterance':<50s} {'expected':<20s} {'got':<20s} {'✓/✗'}")
print("-" * 130)
for options, utt, expected in tests:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: {options}\nUser: {utt}'},
    ]
    raw = generate_ft(messages)
    got = parse_output(raw, valid_letters=options)
    
    if expected is None:
        result = "ambig"
        ambiguous += 1
    elif got == expected or (isinstance(got, list) and isinstance(expected, list) and set(got) == set(expected)):
        result = "✓"
        correct += 1
    else:
        result = "✗"
    
    options_str = str(options)[:23]
    utt_str = utt[:48]
    exp_str = str(expected)[:18]
    got_str = str(got)[:18]
    print(f"{options_str:<25s} {utt_str:<50s} {exp_str:<20s} {got_str:<20s} {result}")

print(f"\ncorrect: {correct}/{len(tests)-ambiguous} (excluding {ambiguous} ambiguous)")

options                   utterance                                          expected             got                  ✓/✗
----------------------------------------------------------------------------------------------------------------------------------
['A', 'B', 'C', 'D']      eh, only B feels right                             ['B']                ['B']                ✓
['A', 'B', 'C', 'D']      throw out everything but the third one             ['C']                ['C']                ✓
['A', 'B', 'C', 'D']      hard pass on all four                              []                   []                   ✓
['A', 'B', 'C', 'D']      A? maybe. dunno.                                   None                 *                    ambig
['A', 'B', 'C', 'D']      the last two are garbage                           ['A', 'B']           ['A', 'B']           ✓
['A', 'B', 'C', 'D']      B leaves D in the dust                             ['B']                ['B']                ✓
['A', 'B', 'C', 

In [24]:
tests = [
    ("E is good", ["A","B","C","D","E"]),
    ("A and E are best", ["A","B","C","D","E"]),
    ("everything except E is wrong", ["A","B","C","D","E"]),
    ("the fifth one is good", ["A","B","C","D","E"]),
    ("not E, not D", ["A","B","C","D","E"]),
    ("F is the best", ["A","B","C","D","E","F"]),
    ("the last one only", ["A","B","C","D","E","F","G"]),
    ("none of these", ["A","B","C","D","E","F"]),
    ("I don't know", ["A","B","C","D","E","F"]),
    ("only G matters", ["A","B","C","D","E","F","G"]),
    ("the last one only", ["A","B","C","D","E","F","G","H","I","J"]),
    ("the last one is out for sure", ["A","B","C","D","E","F","G","H","I","J"]),
]

for utt, opts in tests:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: {opts}\nUser: {utt}'},
    ]
    raw = generate_ft(messages)
    parsed = parse_output(raw, valid_letters=opts)
    print(f"{utt!r:50s} opts={len(opts)} -> raw={raw!r:20s} parsed={parsed}")

'E is good'                                        opts=5 -> raw='["E"]'              parsed=['E']
'A and E are best'                                 opts=5 -> raw='["A", "E"]'         parsed=['A', 'E']
'everything except E is wrong'                     opts=5 -> raw='["E"]'              parsed=['E']
'the fifth one is good'                            opts=5 -> raw='["E"]'              parsed=['E']
'not E, not D'                                     opts=5 -> raw='["A", "B", "C"]'    parsed=['A', 'B', 'C']
'F is the best'                                    opts=6 -> raw='["F"]'              parsed=['F']
'the last one only'                                opts=7 -> raw='["G"]'              parsed=['G']
'none of these'                                    opts=6 -> raw='[]'                 parsed=[]
"I don't know"                                     opts=6 -> raw='"*"'                parsed=*
'only G matters'                                   opts=7 -> raw='["G"]'              parsed=['G']
't

In [25]:
novel_tests = [
    ("eh, only B feels right", ["B"]),
    ("throw out everything but the third one", ["C"]),
    ("hard pass on all four", []),
    ("A? maybe. dunno.", None),  # ambiguous — accept [A] or *
    ("the last two are garbage", ["A","B"]),
    ("B leaves D in the dust", ["B"]),
    ("meh whatever pick anything", "*"),
    ("the second and fourth ones look promising", ["B","D"]),
    ("A is decent, B slightly better, C and D no", ["A","B"]),
    ("give me option C or nothing", ["C"]),
]

correct = 0
print(f"{'utterance':<50s} {'expected':<20s} {'got':<20s} {'result'}")
print("-" * 100)
for utt, expected in novel_tests:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: ["A", "B", "C", "D"]\nUser: {utt}'},
    ]
    raw = generate_ft(messages)
    got = parse_output(raw)
    if expected is None:
        result = "ambig"
    elif got == expected or (isinstance(got, list) and isinstance(expected, list) and set(got) == set(expected)):
        result = "✓"
        correct += 1
    else:
        result = "✗"
    print(f"{utt[:48]:<50s} {str(expected)[:18]:<20s} {str(got)[:18]:<20s} {result}")

print(f"\ncorrect (excluding ambiguous): {correct}/9")

utterance                                          expected             got                  result
----------------------------------------------------------------------------------------------------
eh, only B feels right                             ['B']                ['B']                ✓
throw out everything but the third one             ['C']                ['C']                ✓
hard pass on all four                              []                   []                   ✓
A? maybe. dunno.                                   None                 *                    ambig
the last two are garbage                           ['A', 'B']           ['A', 'B']           ✓
B leaves D in the dust                             ['B']                ['A', 'C']           ✗
meh whatever pick anything                         *                    *                    ✓
the second and fourth ones look promising          ['B', 'D']           ['B', 'D']           ✓
A is decent, B slightly better, C a